# Demo: Khớp ảnh (Weighted Cosine Distance Search)
Notebook này mô phỏng quá trình tìm kiếm một bức ảnh lá (bằng cách lấy một ảnh có sẵn trong DB làm ảnh truy vấn) và sử dụng hàm `search_weighted_cosine` để hiển thị danh sách các ảnh giống nhất, cùng với chi tiết khoảng cách Cosine của từng đặc trưng.

In [1]:
import os
import sys
import pandas as pd

# Thêm thư mục gốc vào sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from db.database import SessionLocal
from db.models import LeafImage
from db.crud import search_weighted_cosine

In [2]:
def run_demo_search():
    db = SessionLocal()
    try:
        # 1. Lấy một ảnh bất kỳ làm ảnh truy vấn (Query Image)
        # Ở đây ta lấy luôn ảnh đầu tiên trong Database để làm mẫu truy vấn
        query_image = db.query(LeafImage).first()
        if not query_image:
            print("Cơ sở dữ liệu trống!")
            return None, None
            
        print(f"Ảnh truy vấn (Query): {query_image.file_name} (ID: {query_image.image_id})")
        
        # 2. Xây dựng dictionary query_vectors từ ảnh này
        query_vectors = {
            "shape": list(query_image.shape_vector),
            "color": list(query_image.color_vector),
            "texture": list(query_image.texture_vector),
            "venation": list(query_image.venation_vector)
        }
        
        # 3. Thực thi tìm kiếm top 5 với hàm search_weighted_cosine
        # Nó sẽ sử dụng trọng số mặc định (mỗi đặc trưng 0.25 hoặc trọng số cấu hình sẵn)
        results = search_weighted_cosine(
            query_vectors=query_vectors,
            top_k=5,
            db=db
        )
        
        # 4. Chuyển đổi kết quả sang Pandas DataFrame
        df = pd.DataFrame(results)
        
        # Sắp xếp lại thứ tự cột cho dễ nhìn
        columns_order = [
            'image_id', 'file_name', 'distance', 
            'shape_dist', 'color_dist', 'texture_dist', 'venation_dist'
        ]
        df = df[columns_order]
        
        # Đổi tên cột cho đẹp để đưa vào báo cáo
        df = df.rename(columns={
            'image_id': 'ID',
            'file_name': 'Tên tệp',
            'distance': 'Tổng khoảng cách',
            'shape_dist': 'KC Hình dạng',
            'color_dist': 'KC Màu sắc',
            'texture_dist': 'KC Kết cấu',
            'venation_dist': 'KC Gân lá'
        })
        
        return query_image.file_name, df
        
    finally:
        db.close()

query_name, result_df = run_demo_search()

Ảnh truy vấn (Query): 1201.jpg (ID: 1201)


In [3]:
# Hiển thị kết quả dưới dạng bảng
if result_df is not None:
    display(result_df)

,ID,Tên tệp,Tổng khoảng cách,KC Hình dạng,KC Màu sắc,KC Kết cấu,KC Gân lá
0,1201,1201.jpg,0.000000,0.000000,0.000000,0.000000,0.000000
1,1232,1232.jpg,0.020478,0.034813,0.001822,0.011108,0.034097
2,1199,1199.jpg,0.023952,0.016690,0.009071,0.024373,0.046080
3,1200,1200.jpg,0.025451,0.027338,0.014420,0.022131,0.038033
4,1229,1229.jpg,0.026585,0.035348,0.000401,0.027467,0.043389
